In [24]:
## "add_message" is a function used to append or combine messages to handel conversation history or chat messages.
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Literal, TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END, START
from langgraph.graph import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver




### TOOLS/FUNCTIONS FOR THE TRAVEL AGENT....

In [19]:

class TravelState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]

@tool
def book_flight(destination: str):
    """Book a flght to the specified destinations."""
    return {"confirmation": "FL12345whacawhace", "destination": destination}

@tool
def book_hotel(location:str):
    """Bool a hotel in the specifed location."""
    return {"confirmation": "HT9845346", "location": location}

def book_car_rental(location: str):
    """Book a car rental in the specified location."""
    return {"confirmation": "gh45234253", "location": location}

tools = [book_flight, book_hotel, book_car_rental]



In [20]:
model = ChatOpenAI(temperature=0, streaming=True)
bound_model= model.bind_tools(tools)

def should_continue(state: TravelState) -> Literal["action", "__end__"]:
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "__end__"
    return "action"

def call_model(state: TravelState):
     response = model.invoke(state["messages"])
     return {"messages": response}


    



### TRAVEL AGENT WORKFLOW...

In [25]:
tool_node = ToolNode(tools)
workflow = StateGraph(TravelState)

workflow.add_node("agent", call_model)
workflow.add_node("action",tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("action", "agent")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)





### CHAT MESSAGE FOR HUMAN AND AI...

In [26]:
config= {"configurable": {"thread_id": "2"}}
input_message = HumanMessage(content="Hello, I want to book a flight to New York?")

for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):

    event["messages"][-1].pretty_print()

input_message = HumanMessage(content="I also want to book a hotel room.")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Hello, I want to book a flight to New York?
================================== Ai Message ==================================

Sure, I can help you with that. Can you please provide me with your departure city, preferred dates of travel, and any other specific preferences you may have for your flight to New York?
================================ Human Message =================================

I also want to book a hotel room.
================================== Ai Message ==================================

Great! Can you please provide me with your preferred check-in and check-out dates, as well as any specific preferences you have for your hotel room in New York? This will help me find the best options for you.
